In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!pip install pytorch-metric-learning -q
print('Ready.')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 5.2 MB/s eta 0:00:00
Ready.


In [ ]:
import os

# Option A: Use Kaggle (paste your credentials)
KAGGLE_USERNAME = 'your_username_here'
KAGGLE_KEY      = 'your_key_here'

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')

!pip install kaggle -q
!kaggle datasets download -d majdouline20/shapenetpart-dataset
!unzip -q shapenetpart-dataset.zip

os.chdir('/content')
print('Dataset ready.')
!find /content/data -name '*.pts' | head -5

Dataset URL: https://www.kaggle.com/datasets/majdouline20/shapenetpart-dataset
License(s): MIT
100% 1.02G/1.02G [00:14<00:00, 75.2MB/s]

Dataset ready.
/content/data/PartAnnotation/04225987/points/48bf45bffab55d7cf14c37b285d25cdf.pts
/content/data/PartAnnotation/04225987/points/b2dc96e7d45797f69a58f604ee2cbadf.pts
/content/data/PartAnnotation/04225987/points/a1d36f02d20d18db6a2979228046d9f9.pts
/content/data/PartAnnotation/04225987/points/54b9d9a3565820e17f267bbeffb219aa.pts
/content/data/PartAnnotation/04225987/points/515f3fab6bee0c40de82f1e4605731da.pts


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import jaccard_score, fbeta_score, precision_score, recall_score
from pytorch_metric_learning.losses import NTXentLoss
from datetime import datetime
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# ─────────────────────────────────────────────────────────────────────────────
# 1. DATASET
# ─────────────────────────────────────────────────────────────────────────────

SYNSET_TO_CLASS = {
    '02691156': 0,  # airplane
    '02773838': 1,  # bag
    '02954340': 2,  # cap
    '02958343': 3,  # car
    '03001627': 4,  # chair
    '03261776': 5,  # earphone
    '03467517': 6,  # guitar
    '03624134': 7,  # knife
    '03636649': 8,  # lamp
    '03642806': 9,  # laptop
    '03790512': 10, # motorbike
    '03797390': 11, # mug
    '03948459': 12, # pistol
    '04099429': 13, # rocket
    '04225987': 14, # skateboard
    '04379243': 15, # table
}
CLASS_TO_NAME = {
    0:'airplane', 1:'bag', 2:'cap', 3:'car', 4:'chair',
    5:'earphone', 6:'guitar', 7:'knife', 8:'lamp', 9:'laptop',
    10:'motorbike', 11:'mug', 12:'pistol', 13:'rocket',
    14:'skateboard', 15:'table'
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3:
                pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    with open(path) as f:
        return np.array([int(l.strip()) for l in f if l.strip()], dtype='int64')

class ShapeNetPart(Dataset):
    def __init__(self, data_root, obj_class=4, partition='train',
                 num_points=1024, train_ratio=0.8):
        self.num_points = num_points

        # find synset
        synset = next(s for s, c in SYNSET_TO_CLASS.items() if c == obj_class)
        syn_dir = os.path.join(data_root, synset)
        pts_dir = os.path.join(syn_dir, 'points')
        seg_dir = os.path.join(syn_dir, 'points_label')

        # build seg map (handle flat or nested)
        seg_map = {}
        if os.path.isdir(seg_dir):
            for entry in os.listdir(seg_dir):
                entry_path = os.path.join(seg_dir, entry)
                if entry.endswith('.seg'):
                    seg_map[entry[:-4]] = entry_path
                elif os.path.isdir(entry_path):
                    for f in os.listdir(entry_path):
                        if f.endswith('.seg') and f[:-4] not in seg_map:
                            seg_map[f[:-4]] = os.path.join(entry_path, f)

        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        valid   = [i for i in all_ids if i in seg_map]

        np.random.seed(42)
        perm  = np.random.permutation(len(valid))
        split = int(len(valid) * train_ratio)
        chosen = [valid[i] for i in (perm[:split] if partition == 'train' else perm[split:])]

        self.samples = [(os.path.join(pts_dir, s+'.pts'), seg_map[s]) for s in chosen]
        print(f'[ShapeNetPart] {CLASS_TO_NAME[obj_class]} | {partition} | {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc), len(seg))
        pc, seg = pc[:n], seg[:n]

        if n >= self.num_points:
            idx_s = np.random.choice(n, self.num_points, replace=False)
        else:
            idx_s = np.random.choice(n, self.num_points, replace=True)
        pc, seg = pc[idx_s], seg[idx_s]

        pc -= pc.mean(0)
        s   = np.max(np.linalg.norm(pc, axis=1))
        if s > 0: pc /= s

        binary = (seg > seg.min()).astype('int64')
        return pc.astype('float32'), binary


# ─────────────────────────────────────────────────────────────────────────────
# 2. DGCNN FEATURE EXTRACTOR (pure PyTorch)
# ─────────────────────────────────────────────────────────────────────────────

def knn(x, k):
    """x: (B,D,N) -> idx: (B,N,k)"""
    xt   = x.permute(0,2,1)              # (B,N,D)
    dist = torch.cdist(xt, xt)           # (B,N,N)
    return dist.topk(k+1, dim=-1, largest=False).indices[:,:,1:]  # (B,N,k)

def get_graph_feature(x, k=20):
    B, D, N = x.shape
    idx     = knn(x, k)                  # (B,N,k)
    idx_base = torch.arange(B, device=x.device).view(-1,1,1) * N
    idx_flat = (idx + idx_base).view(-1)
    xt = x.permute(0,2,1).contiguous()   # (B,N,D)
    nbr = xt.view(B*N,-1)[idx_flat].view(B,N,k,D)
    ctr = xt.view(B,N,1,D).expand(B,N,k,D)
    return torch.cat([nbr-ctr, ctr], dim=3).permute(0,3,1,2).contiguous()  # (B,2D,N,k)

class DGCNN(nn.Module):
    """DGCNN feature extractor — returns per-point features."""
    def __init__(self, k=20, emb_dim=512):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(nn.Conv2d(6,   64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64,  1, bias=False), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, emb_dim, 1, bias=False), nn.BatchNorm1d(emb_dim), nn.LeakyReLU(0.2))
        self.fc    = nn.Sequential(
            nn.Linear(emb_dim*2, 512, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
        )
        self.emb_dim = emb_dim

    def forward(self, x):
        """x: (B,3,N) -> point_feat: (B,emb_dim,N), obj_feat: (B,256)"""
        x1 = self.conv1(get_graph_feature(x,  self.k)).max(-1)[0]   # (B,64,N)
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(-1)[0]   # (B,64,N)
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(-1)[0]   # (B,128,N)
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(-1)[0]   # (B,256,N)

        cat        = torch.cat([x1,x2,x3,x4], dim=1)  # (B,512,N)
        point_feat = self.conv5(cat)                    # (B,emb_dim,N)

        gmp = F.adaptive_max_pool1d(point_feat,1).squeeze(-1)  # (B,emb_dim)
        gap = F.adaptive_avg_pool1d(point_feat,1).squeeze(-1)  # (B,emb_dim)
        obj_feat = self.fc(torch.cat([gmp,gap],1))             # (B,256)

        return point_feat, obj_feat


# ─────────────────────────────────────────────────────────────────────────────
# 3. SAMPLER (learns to pick FG / BG points)
# ─────────────────────────────────────────────────────────────────────────────

class PointSampler(nn.Module):
    """Learns to sample num_out_points from the input cloud."""
    def __init__(self, num_out=256, bottleneck=128, group_size=7):
        super().__init__()
        self.num_out    = num_out
        self.group_size = group_size

        self.enc = nn.Sequential(
            nn.Conv1d(3,  64,  1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, bottleneck, 1), nn.BatchNorm1d(bottleneck), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Linear(bottleneck, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, num_out*3),
        )
        self.sigma = nn.Parameter(torch.tensor(1.0))
        self._simp_loss = None
        self._proj_loss = None

    def forward(self, x):
        """x: (B,N,3) -> simp_pc: (B,M,3), proj_pc: (B,M,3)"""
        B, N, _ = x.shape
        h = self.enc(x.permute(0,2,1)).max(-1)[0]   # (B,bottleneck)
        simp = self.dec(h).view(B, self.num_out, 3)  # (B,M,3)

        # Soft projection onto real points
        dist = torch.cdist(simp, x)                  # (B,M,N)
        g    = min(self.group_size, N)
        topk = dist.topk(g, dim=-1, largest=False)
        kd, ki = topk.values, topk.indices           # (B,M,g)
        nbr  = x.unsqueeze(1).expand(B,self.num_out,N,3)
        ki_e = ki.unsqueeze(-1).expand(B,self.num_out,g,3)
        nbr  = torch.gather(nbr, 2, ki_e)            # (B,M,g,3)
        w    = torch.softmax(-kd / self.sigma.clamp(min=1e-2), -1)  # (B,M,g)
        proj = (w.unsqueeze(-1) * nbr).sum(2)        # (B,M,3)

        # Store losses
        self._simp_loss = dist.min(-1)[0].mean()
        self._proj_loss = ((simp - proj)**2).sum(-1).mean()
        return simp, proj

    def get_simplification_loss(self, pc, simp, n, gamma=1.0):
        return torch.cdist(simp, pc).min(-1)[0].mean()

    def get_projection_loss(self):
        return self._proj_loss if self._proj_loss is not None else torch.tensor(0.0)


class CoSegNet(nn.Module):
    """Full co-segmentation network: FG sampler + BG sampler."""
    def __init__(self, n_fg=256, n_bg=256, group_size=7):
        super().__init__()
        self.fg = PointSampler(n_fg, group_size=group_size)
        self.bg = PointSampler(n_bg, group_size=group_size)


# ─────────────────────────────────────────────────────────────────────────────
# 4. LOSSES
# ─────────────────────────────────────────────────────────────────────────────

def chamfer(pc1, pc2):
    """pc1,pc2: (B,N,3) -> (d1,d2) each (B,N)"""
    d = torch.cdist(pc1, pc2)
    return d.min(2)[0], d.min(1)[0]


def spatial_consistency_loss(point_features, xyz, k=10):
    """
    Professor's extension:
    L_spatial = sum_{i} sum_{j in KNN(i)} ||F_i - F_j||^2

    point_features: (B, D, N)  per-point embeddings
    xyz:            (B, N, 3)  XYZ coordinates
    k:              int        neighbourhood size
    """
    B, D, N = point_features.shape

    # KNN in XYZ space
    dist    = torch.cdist(xyz, xyz)                          # (B,N,N)
    knn_idx = dist.topk(k+1,dim=-1,largest=False).indices[:,:,1:]  # (B,N,k)

    # Gather neighbour features
    feats    = point_features.permute(0,2,1).contiguous()   # (B,N,D)
    flat     = knn_idx.reshape(B,-1)                         # (B,N*k)
    flat_exp = flat.unsqueeze(-1).expand(B, N*k, D)          # (B,N*k,D)
    nbr_f    = torch.gather(feats,1,flat_exp).view(B,N,k,D)  # (B,N,k,D)

    fi   = feats.unsqueeze(2).expand(B,N,k,D)               # (B,N,k,D)
    loss = ((fi - nbr_f)**2).sum(-1).mean()
    return loss


print('All code loaded OK.')
print('Classes defined: ShapeNetPart, DGCNN, CoSegNet, spatial_consistency_loss')

Using device: cuda
All code loaded OK.
Classes defined: ShapeNetPart, DGCNN, CoSegNet, spatial_consistency_loss


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
CFG = {
    'data_root'  : '/content/data/PartAnnotation',  # adjust if needed
    'obj_class'  : 4,        # 4 = chair
    'num_points' : 1024,
    'batch_size' : 8,
    'n_epochs'   : 100,
    'lr'         : 1e-3,
    'n_fg'       : 256,
    'n_bg'       : 256,
    'group_size' : 7,
    'repulsion'  : 0.5,
    'alpha'      : 1.0,      # samplenet loss weight
    'n_pos_pairs': 300,
    'n_neg_pairs': 600,
    'temp_point' : 0.5,
    'temp_obj'   : 0.07,
    # Spatial loss
    'lambda_spatial': 0.1,
    'k_spatial'     : 10,
}

# ── Verify data path ─────────────────────────────────────────────────────────
import os
if not os.path.exists(CFG['data_root']):
    # Try to auto-find
    for root, dirs, files in os.walk('/content/data'):
        if '03001627' in dirs:   # chair synset
            CFG['data_root'] = root
            print(f'Auto-found data_root: {root}')
            break

print('data_root:', CFG['data_root'])
print('exists:', os.path.exists(CFG['data_root']))

# ── Datasets ─────────────────────────────────────────────────────────────────
train_ds = ShapeNetPart(CFG['data_root'], CFG['obj_class'], 'train', CFG['num_points'])
test_ds  = ShapeNetPart(CFG['data_root'], CFG['obj_class'], 'test',  CFG['num_points'])

bs = min(CFG['batch_size'], len(train_ds))
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  drop_last=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=1,  shuffle=False, drop_last=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Test samples: {len(test_ds)}')

# Quick sanity check
pc, lbl = train_ds[0]
print(f'Sample PC shape: {pc.shape} | Label shape: {lbl.shape} | Labels: {set(lbl.tolist())}')

data_root: /content/data/PartAnnotation
exists: True
[ShapeNetPart] chair | train | 5422 samples
[ShapeNetPart] chair | test | 1356 samples
Train batches: 677 | Test samples: 1356
Sample PC shape: (1024, 3) | Label shape: (1024,) | Labels: {0, 1}


In [ ]:
def farthest_point_sample(pc, n):
    """pc: (B,N,3) -> (B,n,3)"""
    B, N, _ = pc.shape
    if n >= N: return pc
    dev = pc.device
    cents = torch.zeros(B,n,dtype=torch.long,device=dev)
    dist  = torch.full((B,N),1e10,device=dev)
    far   = torch.randint(0,N,(B,),device=dev)
    for i in range(n):
        cents[:,i] = far
        c    = pc[torch.arange(B,device=dev),far].unsqueeze(1)
        d    = ((pc-c)**2).sum(-1)
        dist = torch.min(dist,d)
        far  = dist.argmax(-1)
    idx = cents.unsqueeze(-1).expand(B,n,3)
    return torch.gather(pc,1,idx)


def run_training(use_spatial=False, tag='baseline'):
    """
    use_spatial=False  ->  L = L_contrastive            (baseline)
    use_spatial=True   ->  L = L_contrastive + λ*L_spatial  (professor's extension)
    """
    print(f'\n{"="*60}')
    print(f'  Training: {tag.upper()}  |  spatial_loss={use_spatial}')
    print(f'{"="*60}\n')

    os.makedirs(f'/content/results/{tag}', exist_ok=True)

    # Models
    net    = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['group_size']).to(DEVICE)
    feat   = nn.DataParallel(DGCNN(k=20, emb_dim=512)).to(DEVICE)

    opt   = optim.AdamW(net.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, CFG['n_epochs'], eta_min=1e-4)

    pt_loss_fn  = NTXentLoss(temperature=CFG['temp_point'])
    obj_loss_fn = NTXentLoss(temperature=CFG['temp_obj'])

    history = {'iou':[], 'f1':[], 'loss':[]}
    best_f1 = 0.0
    log_path = f'/content/results/{tag}/log.txt'

    def log(msg):
        print(msg)
        with open(log_path,'a') as f: f.write(msg+'\n')

    for epoch in range(CFG['n_epochs']):
        net.train()
        feat.train()
        running = 0.0
        nb      = 0

        for coord, label in train_loader:
            coord = coord.to(DEVICE)           # (B,N,3)
            coord = farthest_point_sample(coord, CFG['num_points'])

            fg_simp, fg_coord = net.fg(coord)  # (B,M,3)
            bg_simp, bg_coord = net.bg(coord)

            # Repulsion loss
            d1, d2 = chamfer(fg_coord, bg_coord)
            rep = (torch.clamp(CFG['repulsion']-d1,min=0).mean() +
                   torch.clamp(CFG['repulsion']-d2,min=0).mean())

            # Features  (B, emb_dim, M)
            fg_pt_feat, fg_obj = feat(fg_coord.permute(0,2,1).contiguous())
            bg_pt_feat, bg_obj = feat(bg_coord.permute(0,2,1).contiguous())

            # SampleNet loss
            fg_sl = net.fg.get_simplification_loss(coord, fg_simp, CFG['n_fg'])
            fg_pl = net.fg.get_projection_loss()
            bg_sl = net.bg.get_simplification_loss(coord, bg_simp, CFG['n_bg'])
            bg_pl = net.bg.get_projection_loss()
            samp_loss = CFG['alpha'] * (fg_sl + fg_pl + bg_sl + bg_pl)

            # Point-level contrastive loss
            B = coord.shape[0]
            pt_losses = []
            for i in range(B):
                of = fg_pt_feat[i].permute(1,0)  # (M, D)
                bf = bg_pt_feat[i].permute(1,0)
                emb = torch.cat([of, bf], 0)
                lbl = torch.cat([
                    torch.zeros(of.shape[0]),
                    torch.arange(1, bf.shape[0]+1)
                ]).long().to(DEVICE)
                pos = torch.randint(0,of.shape[0],(CFG['n_pos_pairs'],2),device=DEVICE)
                neg = torch.randint(0,bf.shape[0],(CFG['n_neg_pairs'],2),device=DEVICE)
                neg[:,1] += of.shape[0]
                tup = (pos[:,0], pos[:,1], neg[:,0], neg[:,1])
                pt_losses.append(pt_loss_fn(emb, lbl, tup))
            pt_loss = torch.stack(pt_losses).mean()

            # Object-level contrastive loss
            emb = torch.cat([fg_obj, bg_obj], 0)
            lbl = torch.cat([
                torch.zeros(fg_obj.shape[0]),
                torch.arange(1, bg_obj.shape[0]+1)
            ]).long().to(DEVICE)
            obj_loss = obj_loss_fn(emb, lbl)

            # ── TOTAL LOSS ────────────────────────────────────────────────────
            if use_spatial:
                # Professor's extension: add spatial consistency loss
                sp_loss = spatial_consistency_loss(
                    fg_pt_feat, fg_coord, k=CFG['k_spatial']
                )
                loss = pt_loss + obj_loss + samp_loss + rep + CFG['lambda_spatial'] * sp_loss
            else:
                # Baseline: contrastive only
                loss = pt_loss + obj_loss + samp_loss + rep

            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item()
            nb      += 1

        sched.step()

        # ── VALIDATION ───────────────────────────────────────────────────────
        net.eval()
        feat.eval()
        y_preds, y_trues = [], []

        with torch.no_grad():
            for coord, label in test_loader:
                coord = coord.to(DEVICE)
                coord, fps_idx = farthest_point_sample(coord, CFG['num_points']), None

                _, fg_coord = net.fg(coord)  # (1,M,3)

                # KNN: find input points near FG samples
                dist    = torch.cdist(coord, fg_coord)   # (1,N,M)
                min_d   = dist.min(-1)[0].squeeze()      # (N,)
                thresh  = min_d.topk(CFG['n_fg'], largest=False).values[-1]
                pred    = (min_d <= thresh).long().cpu().numpy()

                true = label.squeeze().numpy()
                y_preds.append(pred)
                y_trues.append(true)

        yp = np.concatenate(y_preds)
        yt = np.concatenate(y_trues)

        iou = jaccard_score(yt, yp, zero_division=0)
        f1  = fbeta_score(yt,  yp, beta=0.75, zero_division=0)
        pre = precision_score(yt, yp, zero_division=0)
        rec = recall_score(yt,    yp, zero_division=0)

        history['iou'].append(iou)
        history['f1'].append(f1)
        history['loss'].append(running/nb)

        msg = (f'[{tag}] Epoch {epoch:03d} | loss {running/nb:.4f} | '
               f'IOU {iou:.4f} | F1 {f1:.4f} | P {pre:.3f} | R {rec:.3f}')
        log(msg)

        if f1 > best_f1:
            best_f1 = f1
            torch.save(net.state_dict(), f'/content/results/{tag}/model_best.pt')
            log(f'  --> New best F1={best_f1:.4f}')

    log(f'\nFINISHED [{tag}] | Best F1={best_f1:.4f} | Best IOU={max(history["iou"]):.4f}')
    return history


print('Training function defined. Ready to run.')

Training function defined. Ready to run.


In [ ]:
# L = L_contrastive  (no spatial loss)
history_baseline = run_training(use_spatial=False, tag='baseline')


  Training: BASELINE  |  spatial_loss=False

[baseline] Epoch 000 | loss 5.8142 | IOU 0.1331 | F1 0.2473 | P 0.286 | R 0.199
  --> New best F1=0.2473
[baseline] Epoch 001 | loss 5.6502 | IOU 0.2269 | F1 0.3894 | P 0.451 | R 0.313
  --> New best F1=0.3894
[baseline] Epoch 002 | loss 5.5975 | IOU 0.3191 | F1 0.5096 | P 0.590 | R 0.410
  --> New best F1=0.5096
[baseline] Epoch 003 | loss 5.5556 | IOU 0.2938 | F1 0.4782 | P 0.554 | R 0.385
[baseline] Epoch 004 | loss 5.5325 | IOU 0.3222 | F1 0.5132 | P 0.594 | R 0.413
  --> New best F1=0.5132
[baseline] Epoch 005 | loss 5.4827 | IOU 0.3489 | F1 0.5446 | P 0.630 | R 0.439
  --> New best F1=0.5446
[baseline] Epoch 006 | loss 5.4536 | IOU 0.3590 | F1 0.5564 | P 0.644 | R 0.448
  --> New best F1=0.5564
[baseline] Epoch 007 | loss 5.4415 | IOU 0.3786 | F1 0.5784 | P 0.670 | R 0.465
  --> New best F1=0.5784
[baseline] Epoch 008 | loss 5.4106 | IOU 0.3515 | F1 0.5479 | P 0.635 | R 0.441
[baseline] Epoch 009 | loss 5.4014 | IOU 0.3540 | F1 0.5506

KeyboardInterrupt: 

In [ ]:
# L = L_contrastive + lambda * L_spatial
history_spatial = run_training(use_spatial=True, tag='spatial_loss')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Baseline vs Spatial Loss — ShapeNet Part (Chair)', fontsize=14, fontweight='bold')

epochs = range(1, CFG['n_epochs']+1)

for ax, key, title in zip(axes, ['iou','f1','loss'], ['IOU','F1 Score','Loss']):
    ax.plot(epochs, history_baseline[key], label='Baseline',      color='steelblue',  linewidth=2)
    ax.plot(epochs, history_spatial[key],  label='+Spatial Loss',  color='darkorange', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\n' + '='*55)
print(f'{"":20} {"Baseline":>15} {"Spatial Loss":>15}')
print('='*55)
print(f'{"Best IOU":20} {max(history_baseline["iou"]):>15.4f} {max(history_spatial["iou"]):>15.4f}')
print(f'{"Best F1":20} {max(history_baseline["f1"]):>15.4f} {max(history_spatial["f1"]):>15.4f}')
print(f'{"Final Loss":20} {history_baseline["loss"][-1]:>15.4f} {history_spatial["loss"][-1]:>15.4f}')
print('='*55)
diff_iou = max(history_spatial['iou']) - max(history_baseline['iou'])
print(f'IOU improvement from spatial loss: {diff_iou:+.4f}')

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/BTP_coseg_results'
os.makedirs(dest, exist_ok=True)
shutil.copytree('/content/results', dest, dirs_exist_ok=True)
print(f'Saved to: {dest}')
print('Contents:')
for f in os.listdir(dest):
    print(' ', f)

In [ ]:
import time
for i in range(200):
    time.sleep(60)
    print(f'Alive: {i} min')